# Diffusion Models for High-Resolution Image Generation & Reconstruction
Assignment 04 — Generative AI (AI4009)

DDPM trained from scratch on CelebA-HQ. Built with base PyTorch only (no diffusers, no pretrained weights).

## 1. Environment Setup

In [ ]:
import os, math, time, random, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.amp import autocast, GradScaler
import torchvision
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
print("device:", device, "| gpus:", n_gpu)
if torch.cuda.is_available():
    for i in range(n_gpu):
        print(" ", i, torch.cuda.get_device_name(i))

In [ ]:
def gather_all_images(root="/kaggle/input"):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        files.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    return sorted(files)

all_files = gather_all_images("/kaggle/input")
print("total images found:", len(all_files))
assert len(all_files) > 1000, "dataset not found, attach celebahq256-images-only on kaggle"
print("sample path:", all_files[0])

## 2. Configuration

In [ ]:
class Config:
    image_size   = 128
    channels     = 3
    batch_size   = 32
    num_workers  = 2
    subset_size  = 20000
    epochs       = 60
    lr           = 2e-4
    timesteps    = 500
    base_ch      = 64
    ch_mults     = (1, 2, 4)
    ema_decay    = 0.999
    grad_clip    = 1.0
    sample_every = 5
    save_dir     = "/kaggle/working"

cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)

## 3. Data Pipeline

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, files, image_size):
        self.files = files
        self.tf = transforms.Compose([
            transforms.Resize(image_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        return self.tf(img)

subset_files = all_files[:cfg.subset_size]
train_ds = FaceDataset(subset_files, cfg.image_size)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
print("training samples:", len(train_ds), "| batches/epoch:", len(train_loader))

In [ ]:
def show_batch(batch, n=8, title=None):
    batch = (batch[:n].clamp(-1, 1) + 1) / 2
    grid = torchvision.utils.make_grid(batch, nrow=n)
    plt.figure(figsize=(n*1.6, 2))
    plt.imshow(grid.permute(1,2,0).cpu().numpy())
    plt.axis("off")
    if title: plt.title(title)
    plt.show()

real = next(iter(train_loader))
show_batch(real, n=8, title="real samples")

## 4. Forward Diffusion (Noising)

In [ ]:
def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

class Diffusion:
    def __init__(self, T, device, schedule="linear"):
        self.T = T
        if schedule == "linear":
            self.betas = linear_beta_schedule(T).to(device)
        else:
            self.betas = cosine_beta_schedule(T).to(device)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bar = torch.sqrt(self.alpha_bar)
        self.sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - self.alpha_bar)
        self.device = device

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sa = self.sqrt_alpha_bar[t].view(-1,1,1,1)
        som = self.sqrt_one_minus_alpha_bar[t].view(-1,1,1,1)
        return sa * x0 + som * noise, noise

def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    x = torch.linspace(0, T, steps)
    f = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = f / f[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)

diffusion = Diffusion(cfg.timesteps, device, schedule="linear")
print("betas range:", diffusion.betas.min().item(), diffusion.betas.max().item())

In [ ]:
sample_img = real[0:1].to(device)
steps_to_show = [0, cfg.timesteps//5, cfg.timesteps*2//5, cfg.timesteps*3//5, cfg.timesteps*4//5, cfg.timesteps-1]

fig, axes = plt.subplots(1, len(steps_to_show), figsize=(len(steps_to_show)*2, 2))
for ax, t_val in zip(axes, steps_to_show):
    t = torch.tensor([t_val], device=device)
    xt, _ = diffusion.q_sample(sample_img, t)
    img = (xt[0].clamp(-1,1) + 1) / 2
    ax.imshow(img.permute(1,2,0).cpu().numpy()); ax.set_title(f"t={t_val}"); ax.axis("off")
plt.suptitle("forward diffusion steps")
plt.tight_layout(); plt.show()

## 5. U-Net Architecture

In [ ]:
class SinusoidalTimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        args = t[:, None].float() * freqs[None]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim, groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class SelfAttention(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__()
        self.heads = heads
        self.norm = nn.GroupNorm(8, ch)
        self.qkv = nn.Conv2d(ch, ch*3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv(self.norm(x)).reshape(b, 3, self.heads, c//self.heads, h*w)
        q, k, v = qkv[:,0], qkv[:,1], qkv[:,2]
        attn = torch.softmax(q.transpose(-1,-2) @ k / math.sqrt(c//self.heads), dim=-1)
        out = (v @ attn.transpose(-1,-2)).reshape(b, c, h, w)
        return x + self.proj(out)

class Down(nn.Module):
    def __init__(self, ch): super().__init__(); self.op = nn.Conv2d(ch, ch, 4, 2, 1)
    def forward(self, x): return self.op(x)

class Up(nn.Module):
    def __init__(self, ch): super().__init__(); self.op = nn.ConvTranspose2d(ch, ch, 4, 2, 1)
    def forward(self, x): return self.op(x)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, ch_mults=(1,2,4), t_dim=256):
        super().__init__()
        self.t_embed = nn.Sequential(
            SinusoidalTimeEmbed(base_ch),
            nn.Linear(base_ch, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        chs = [base_ch * m for m in ch_mults]
        self.in_conv = nn.Conv2d(in_ch, chs[0], 3, padding=1)

        self.down_blocks = nn.ModuleList()
        self.downs = nn.ModuleList()
        prev = chs[0]
        for i, c in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(prev, c, t_dim),
                ResBlock(c, c, t_dim),
            ]))
            if i != len(chs) - 1:
                self.downs.append(Down(c))
            prev = c

        self.mid1 = ResBlock(chs[-1], chs[-1], t_dim)
        self.mid_attn = SelfAttention(chs[-1])
        self.mid2 = ResBlock(chs[-1], chs[-1], t_dim)

        self.up_blocks = nn.ModuleList()
        self.ups = nn.ModuleList()
        rev_chs = list(reversed(chs))
        for i, c in enumerate(rev_chs):
            in_c = prev + c
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_c, c, t_dim),
                ResBlock(c, c, t_dim),
            ]))
            if i != len(rev_chs) - 1:
                self.ups.append(Up(c))
            prev = c

        self.out_norm = nn.GroupNorm(8, chs[0])
        self.out_conv = nn.Conv2d(chs[0], in_ch, 3, padding=1)

    def forward(self, x, t):
        te = self.t_embed(t)
        h = self.in_conv(x)
        skips = []
        for i, blocks in enumerate(self.down_blocks):
            for b in blocks:
                h = b(h, te)
            skips.append(h)
            if i != len(self.down_blocks) - 1:
                h = self.downs[i](h)

        h = self.mid1(h, te)
        h = self.mid_attn(h)
        h = self.mid2(h, te)

        for i, blocks in enumerate(self.up_blocks):
            skip = skips.pop()
            h = torch.cat([h, skip], dim=1)
            for b in blocks:
                h = b(h, te)
            if i != len(self.up_blocks) - 1:
                h = self.ups[i](h)

        return self.out_conv(F.silu(self.out_norm(h)))

model = UNet(in_ch=cfg.channels, base_ch=cfg.base_ch, ch_mults=cfg.ch_mults).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"unet parameters: {n_params/1e6:.2f}M")

## 6. Training

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items() if v.dtype.is_floating_point}
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
    def apply_to(self, model):
        model.load_state_dict({**model.state_dict(), **self.shadow}, strict=False)

if n_gpu > 1:
    train_model = nn.DataParallel(model)
else:
    train_model = model

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs * len(train_loader))
scaler = GradScaler("cuda")
ema = EMA(model, decay=cfg.ema_decay)

loss_history = []

In [ ]:
def train_one_epoch(epoch):
    train_model.train()
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{cfg.epochs}")
    epoch_losses = []
    for x in pbar:
        x = x.to(device, non_blocking=True)
        b = x.size(0)
        t = torch.randint(0, cfg.timesteps, (b,), device=device)
        x_t, noise = diffusion.q_sample(x, t)

        optimizer.zero_grad()
        with autocast("cuda"):
            pred = train_model(x_t, t)
            loss = F.mse_loss(pred, noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(model)

        epoch_losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return float(np.mean(epoch_losses))

In [ ]:
start = time.time()
for epoch in range(cfg.epochs):
    avg = train_one_epoch(epoch)
    loss_history.append(avg)
    print(f"epoch {epoch+1}: avg loss = {avg:.4f}")
elapsed = (time.time() - start) / 60
print(f"training done in {elapsed:.1f} min")
torch.save(model.state_dict(), os.path.join(cfg.save_dir, "ddpm_unet.pt"))
torch.save(ema.shadow, os.path.join(cfg.save_dir, "ddpm_unet_ema.pt"))

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(loss_history, marker="o")
plt.xlabel("epoch"); plt.ylabel("MSE loss")
plt.title("training loss")
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(cfg.save_dir, "loss_curve.png"), dpi=120, bbox_inches="tight")
plt.show()

## 7. Reverse Process — Sampling

In [ ]:
@torch.no_grad()
def ddpm_sample(model_eval, diff, n=4, image_size=128, return_intermediates=False, intermediates_at=None):
    model_eval.eval()
    x = torch.randn(n, 3, image_size, image_size, device=device)
    inters = {}
    if intermediates_at is None:
        intermediates_at = []
    for t in reversed(range(diff.T)):
        ts = torch.full((n,), t, device=device, dtype=torch.long)
        with autocast("cuda"):
            pred_noise = model_eval(x, ts)
        beta = diff.betas[t]
        alpha = diff.alphas[t]
        alpha_bar = diff.alpha_bar[t]
        coef = (1 - alpha) / torch.sqrt(1 - alpha_bar)
        mean = (1 / torch.sqrt(alpha)) * (x - coef * pred_noise)
        if t > 0:
            noise = torch.randn_like(x)
            x = mean + torch.sqrt(beta) * noise
        else:
            x = mean
        if t in intermediates_at:
            inters[t] = x.clone()
    if return_intermediates:
        return x.clamp(-1,1), inters
    return x.clamp(-1,1)

eval_model = UNet(in_ch=cfg.channels, base_ch=cfg.base_ch, ch_mults=cfg.ch_mults).to(device)
eval_model.load_state_dict({**model.state_dict(), **ema.shadow}, strict=False)

In [ ]:
generated = ddpm_sample(eval_model, diffusion, n=8, image_size=cfg.image_size)
show_batch(generated, n=8, title="ddpm generated samples")

## 8. Visualizing Reverse Diffusion

In [ ]:
marks = sorted(set([cfg.timesteps-1, int(cfg.timesteps*0.8), int(cfg.timesteps*0.6),
                     int(cfg.timesteps*0.4), int(cfg.timesteps*0.2), 0]))
out, inter = ddpm_sample(eval_model, diffusion, n=4, image_size=cfg.image_size,
                         return_intermediates=True, intermediates_at=marks)

fig, axes = plt.subplots(4, len(marks), figsize=(len(marks)*1.8, 4*1.8))
for col, t_val in enumerate(reversed(marks)):
    imgs = (inter[t_val].clamp(-1,1) + 1) / 2
    for row in range(4):
        axes[row, col].imshow(imgs[row].permute(1,2,0).cpu().numpy())
        axes[row, col].axis("off")
        if row == 0: axes[row, col].set_title(f"t={t_val}")
plt.suptitle("reverse diffusion progress")
plt.tight_layout(); plt.show()

## 9. Image Reconstruction (Core Task)
Take a target image, partially noise it, then denoise back so the result resembles the target.

In [ ]:
@torch.no_grad()
def reconstruct(model_eval, diff, target, t_start=None):
    model_eval.eval()
    if t_start is None:
        t_start = diff.T - 1
    target = target.to(device)
    t = torch.full((target.size(0),), t_start, device=device, dtype=torch.long)
    x, _ = diff.q_sample(target, t)
    for ti in reversed(range(t_start + 1)):
        ts = torch.full((target.size(0),), ti, device=device, dtype=torch.long)
        with autocast("cuda"):
            pred_noise = model_eval(x, ts)
        beta = diff.betas[ti]; alpha = diff.alphas[ti]; alpha_bar = diff.alpha_bar[ti]
        coef = (1 - alpha) / torch.sqrt(1 - alpha_bar)
        mean = (1 / torch.sqrt(alpha)) * (x - coef * pred_noise)
        if ti > 0:
            x = mean + torch.sqrt(beta) * torch.randn_like(x)
        else:
            x = mean
    return x.clamp(-1, 1)

target_batch = next(iter(train_loader))[:4].to(device)
recon = reconstruct(eval_model, diffusion, target_batch, t_start=int(cfg.timesteps*0.5))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    t_img = (target_batch[i].clamp(-1,1) + 1) / 2
    r_img = (recon[i].clamp(-1,1) + 1) / 2
    axes[0, i].imshow(t_img.permute(1,2,0).cpu().numpy()); axes[0, i].axis("off")
    axes[1, i].imshow(r_img.permute(1,2,0).cpu().numpy()); axes[1, i].axis("off")
axes[0, 0].set_ylabel("target"); axes[1, 0].set_ylabel("reconstructed")
axes[0, 0].set_title("target"); axes[1, 0].set_title("reconstructed")
plt.suptitle("target vs reconstruction")
plt.tight_layout(); plt.show()

## 10. Quantitative Evaluation — PSNR & SSIM

In [ ]:
def psnr(a, b, max_val=1.0):
    a = a.float(); b = b.float()
    mse = F.mse_loss(a, b)
    if mse < 1e-12: return float("inf")
    return (20 * torch.log10(torch.tensor(max_val)) - 10 * torch.log10(mse)).item()

def ssim(a, b, C1=0.01**2, C2=0.03**2, win=11):
    a = a.unsqueeze(0) if a.dim()==3 else a
    b = b.unsqueeze(0) if b.dim()==3 else b
    pad = win // 2
    kernel = torch.ones(1, 1, win, win, device=a.device) / (win * win)
    def _filter(x):
        c = x.shape[1]
        k = kernel.repeat(c, 1, 1, 1)
        return F.conv2d(x, k, padding=pad, groups=c)
    mu_a = _filter(a); mu_b = _filter(b)
    mu_a2 = mu_a*mu_a; mu_b2 = mu_b*mu_b; mu_ab = mu_a*mu_b
    sigma_a2 = _filter(a*a) - mu_a2
    sigma_b2 = _filter(b*b) - mu_b2
    sigma_ab = _filter(a*b) - mu_ab
    num = (2*mu_ab + C1) * (2*sigma_ab + C2)
    den = (mu_a2 + mu_b2 + C1) * (sigma_a2 + sigma_b2 + C2)
    return (num / den).mean().item()

t01 = (target_batch.clamp(-1,1) + 1) / 2
r01 = (recon.clamp(-1,1) + 1) / 2
psnr_vals, ssim_vals = [], []
for i in range(t01.size(0)):
    psnr_vals.append(psnr(t01[i], r01[i]))
    ssim_vals.append(ssim(t01[i], r01[i]))
print(f"PSNR per image: {[f'{v:.2f}' for v in psnr_vals]}  mean: {np.mean(psnr_vals):.2f} dB")
print(f"SSIM per image: {[f'{v:.3f}' for v in ssim_vals]}  mean: {np.mean(ssim_vals):.3f}")

## 11. Side-by-Side Comparison & 5+ Generated

In [ ]:
five = ddpm_sample(eval_model, diffusion, n=5, image_size=cfg.image_size)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for i in range(5):
    img = (five[i].clamp(-1,1) + 1) / 2
    axes[i].imshow(img.permute(1,2,0).cpu().numpy()); axes[i].axis("off")
    axes[i].set_title(f"gen {i+1}")
plt.suptitle("5 generated images from pure noise")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    t_img = (target_batch[i].clamp(-1,1) + 1) / 2
    g_img = (five[i % 5].clamp(-1,1) + 1) / 2
    axes[0, i].imshow(t_img.permute(1,2,0).cpu().numpy()); axes[0, i].set_title("target"); axes[0, i].axis("off")
    axes[1, i].imshow(g_img.permute(1,2,0).cpu().numpy()); axes[1, i].set_title("generated"); axes[1, i].axis("off")
plt.suptitle("target vs generated (side by side)")
plt.tight_layout(); plt.show()

## 12. Gradio App Deployment

In [ ]:
try:
    import gradio
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
import gradio as gr

@torch.no_grad()
def generate_app(num_steps_shown=6, seed_val=0):
    torch.manual_seed(int(seed_val))
    eval_model.eval()
    n = 1
    x = torch.randn(n, 3, cfg.image_size, cfg.image_size, device=device)
    marks = np.linspace(cfg.timesteps-1, 0, num_steps_shown).astype(int).tolist()
    frames = []
    for t in reversed(range(cfg.timesteps)):
        ts = torch.full((n,), t, device=device, dtype=torch.long)
        with autocast("cuda"):
            pn = eval_model(x, ts)
        beta = diffusion.betas[t]; alpha = diffusion.alphas[t]; alpha_bar = diffusion.alpha_bar[t]
        coef = (1-alpha) / torch.sqrt(1-alpha_bar)
        mean = (1/torch.sqrt(alpha)) * (x - coef*pn)
        if t > 0:
            x = mean + torch.sqrt(beta)*torch.randn_like(x)
        else:
            x = mean
        if t in marks:
            img = ((x[0].clamp(-1,1)+1)/2).permute(1,2,0).cpu().numpy()
            frames.append((img, f"t={t}"))
    final = ((x[0].clamp(-1,1)+1)/2).permute(1,2,0).cpu().numpy()
    return final, frames

with gr.Blocks(title="DDPM Generator") as demo:
    gr.Markdown("### DDPM image generation from pure noise")
    with gr.Row():
        steps_slider = gr.Slider(3, 12, value=6, step=1, label="intermediate frames")
        seed_box = gr.Number(value=0, label="seed")
    btn = gr.Button("generate")
    final_img = gr.Image(label="final image")
    gallery = gr.Gallery(label="denoising steps", columns=6)
    btn.click(generate_app, inputs=[steps_slider, seed_box], outputs=[final_img, gallery])

demo.launch(share=True, debug=False)

## BONUS TASK
Improved sampling (DDIM), cosine noise schedule comparison, and a 256×256 generation pass.

### Bonus 1 — DDIM Faster Sampling

In [ ]:
@torch.no_grad()
def ddim_sample(model_eval, diff, n=4, image_size=128, ddim_steps=50, eta=0.0):
    model_eval.eval()
    x = torch.randn(n, 3, image_size, image_size, device=device)
    step_seq = np.linspace(0, diff.T - 1, ddim_steps).astype(int)[::-1]
    for i, t in enumerate(step_seq):
        ts = torch.full((n,), int(t), device=device, dtype=torch.long)
        with autocast("cuda"):
            pred_noise = model_eval(x, ts)
        a_bar = diff.alpha_bar[t]
        a_bar_prev = diff.alpha_bar[step_seq[i+1]] if i+1 < len(step_seq) else torch.tensor(1.0, device=device)
        x0_pred = (x - torch.sqrt(1 - a_bar) * pred_noise) / torch.sqrt(a_bar)
        x0_pred = x0_pred.clamp(-1, 1)
        sigma = eta * torch.sqrt((1 - a_bar_prev) / (1 - a_bar) * (1 - a_bar / a_bar_prev))
        dir_xt = torch.sqrt(1 - a_bar_prev - sigma**2) * pred_noise
        x = torch.sqrt(a_bar_prev) * x0_pred + dir_xt
        if eta > 0:
            x = x + sigma * torch.randn_like(x)
    return x.clamp(-1, 1)

t0 = time.time(); ddpm_imgs = ddpm_sample(eval_model, diffusion, n=5, image_size=cfg.image_size); ddpm_time = time.time()-t0
t0 = time.time(); ddim_imgs = ddim_sample(eval_model, diffusion, n=5, image_size=cfg.image_size, ddim_steps=50); ddim_time = time.time()-t0
print(f"DDPM ({cfg.timesteps} steps): {ddpm_time:.1f}s   DDIM (50 steps): {ddim_time:.1f}s   speedup: {ddpm_time/max(ddim_time,1e-6):.1f}x")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    a = (ddpm_imgs[i].clamp(-1,1)+1)/2
    b = (ddim_imgs[i].clamp(-1,1)+1)/2
    axes[0, i].imshow(a.permute(1,2,0).cpu().numpy()); axes[0,i].set_title("DDPM"); axes[0,i].axis("off")
    axes[1, i].imshow(b.permute(1,2,0).cpu().numpy()); axes[1,i].set_title("DDIM"); axes[1,i].axis("off")
plt.suptitle("ddpm vs ddim (faster sampling)")
plt.tight_layout(); plt.show()

### Bonus 2 — Cosine Noise Schedule

In [ ]:
diff_cosine = Diffusion(cfg.timesteps, device, schedule="cosine")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(diffusion.betas.cpu().numpy(), label="linear")
ax[0].plot(diff_cosine.betas.cpu().numpy(), label="cosine")
ax[0].set_title("beta schedule"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(diffusion.alpha_bar.cpu().numpy(), label="linear")
ax[1].plot(diff_cosine.alpha_bar.cpu().numpy(), label="cosine")
ax[1].set_title("alpha_bar"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

steps_to_show = [0, cfg.timesteps//4, cfg.timesteps//2, 3*cfg.timesteps//4, cfg.timesteps-1]
fig, axes = plt.subplots(2, len(steps_to_show), figsize=(len(steps_to_show)*2, 4))
for col, t_val in enumerate(steps_to_show):
    t = torch.tensor([t_val], device=device)
    xL, _ = diffusion.q_sample(sample_img, t)
    xC, _ = diff_cosine.q_sample(sample_img, t)
    axes[0, col].imshow(((xL[0].clamp(-1,1)+1)/2).permute(1,2,0).cpu().numpy()); axes[0,col].axis("off"); axes[0,col].set_title(f"linear t={t_val}")
    axes[1, col].imshow(((xC[0].clamp(-1,1)+1)/2).permute(1,2,0).cpu().numpy()); axes[1,col].axis("off"); axes[1,col].set_title(f"cosine t={t_val}")
plt.suptitle("noising under different schedules")
plt.tight_layout(); plt.show()

### Bonus 3 — 256×256 Generation
Sample directly at higher resolution. The trained 128 model is convolutional so it can run at 256, results will be coarser but demonstrate scalability.

In [ ]:
hires = ddim_sample(eval_model, diffusion, n=4, image_size=256, ddim_steps=80)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for i in range(4):
    img = (hires[i].clamp(-1,1)+1)/2
    axes[i].imshow(img.permute(1,2,0).cpu().numpy()); axes[i].axis("off"); axes[i].set_title("256x256")
plt.suptitle("bonus: higher-resolution sampling")
plt.tight_layout(); plt.show()

### Bonus 4 — Conclusion
DDPM trained from scratch reaches stable loss, generates coherent face structures, and DDIM gives a roughly 10x sampling speedup with comparable quality. Cosine schedule destroys structure more gradually than linear in early steps. Higher-resolution sampling is possible since the U-Net is fully convolutional.